#Orange Problem: Multimodal Fine Tuning with SLM

#Team: Temporal_Trio

# Approach

The goal of this project was to fine-tune a multimodal Small Language Model (SLM) to generate natural language descriptions of mobile UI screenshots.

### Dataset Selection

Two datasets were available for this assignment:

- ChartQA
- RICO Screen2Words

We selected **RICO Screen2Words** because it directly pairs UI screenshots with human-written descriptions, making it suitable for image-to-text generation tasks.

### Model Selection

We used the pretrained **BLIP (Bootstrapping Language Image Pretraining)** model:

Salesforce/blip-image-captioning-base

BLIP is a vision-language model that combines:

- a **vision encoder** to process images
- a **language decoder** to generate captions

This architecture makes it well suited for multimodal caption generation.

### Training Pipeline

The full training pipeline consisted of:

1. Loading the RICO Screen2Words dataset from Hugging Face
2. Preprocessing images and captions using the BLIP processor
3. Creating training and validation splits
4. Fine-tuning the BLIP model using the Hugging Face Trainer API
5. Uploading the trained model to the Hugging Face Hub
6. Evaluating the model on unseen UI screenshots
7. Inference on Hugging Face Model


### Training Configuration

| Parameter | Value |
|------|------|
Batch size | 4 |
Epochs | 3 |
Learning rate | 5e-5 |
Precision | fp16 |

Mixed precision training was used to ensure the model runs efficiently on **T4 GPU hardware**.<br> <br><br>  
  




## Hardware Constraint

The assignment requires the full pipeline to run on **T4 GPU compute** (such as Google Colab or Kaggle).

To satisfy this constraint:

- We use **BLIP base model**, which is lightweight enough for T4 GPUs.
- We use **small batch sizes** to fit within GPU memory.
- We enable **FP16 mixed precision training** to reduce memory usage.
<br> <br><br>  

# Assumptions

Several assumptions were made during the development of this project.

### Single Caption per Screen

Each dataset entry contains multiple captions.  
For simplicity, only the **first caption** was used as the training target.

### Caption Generalization

The model is expected to generate **semantically correct descriptions**, even if the wording differs from the ground truth.

### Hardware Constraints

The training pipeline was designed to run on **T4 GPU compute**, as required by the assignment.  
This influenced the choice of:

- model size
- batch size
- training configuration.

### Focus on UI Summarization

The model is intended specifically for **mobile UI screenshots** and may not perform well on general scene images.
<br><br><br>

# Dataset Selection

## RICO Screen2Words Dataset

For this project we chose the **RICO Screen2Words dataset**, available on Hugging Face:

https://huggingface.co/datasets/rootsautomation/RICO-Screen2Words

# Reasons for Choosing RICO Screen2Words

## 1. Strong Alignment with Multimodal Captioning Tasks

The RICO Screen2Words dataset directly pairs:
UI Screenshot → Natural Language Description


This makes it naturally suited for **image-to-text generation tasks**, which involve understanding visual structure and producing descriptive language.

---

## 2. Real-World Mobile Interface Data

The dataset consists of **screenshots from real Android applications collected from the Google Play Store**.

This provides realistic training data representing:

- login screens
- search pages
- navigation interfaces
- product pages
- messaging screens

As a result, the model learns from **authentic UI layouts rather than synthetic data**.

---

## 3. Human-Written Summaries

Captions in the dataset were written by **professional annotators**.

These descriptions summarize the **function and layout of the screen**, not just the visual objects.

This improves training quality because the model learns to generate **meaningful descriptions of interface functionality**.

---

## 4. Designed Specifically for Multimodal UI Understanding

The dataset was created to support research in tasks such as:

- automatic screen summarization
- screen indexing
- language-based UI retrieval
- conversational mobile interfaces
- accessibility tools for screen readers

Because the dataset explicitly connects **visual UI structures with language descriptions**, it is well suited for multimodal learning.

---

## 5. Rich Metadata and Structured Annotations

In addition to images and captions, the dataset includes:

- application name
- app category
- download statistics
- view hierarchy information
- semantic UI annotations

Although this project focuses only on images and captions, these additional fields make the dataset valuable for **future multimodal research tasks**.

---

# Dataset Statistics

The dataset includes:

- ~22,000 UI screens
- screenshots from ~9,800 mobile applications
- captions written by **85 professional annotators**

This provides a sufficiently large dataset for training multimodal models.

---

# Final Decision

Based on the analysis above, **RICO Screen2Words was chosen over ChartQA** because:

- it directly supports **image-to-text generation**
- it provides **human-written summaries of visual interfaces**
- it represents **real-world mobile application screens**
- it aligns well with multimodal UI understanding tasks

Therefore, it provides a more appropriate dataset for demonstrating **multimodal fine-tuning with a Small Language Model (SLM)**.


 # 1.Install Dependencies

In [1]:
!pip install transformers datasets accelerate peft bitsandbytes torchvision --quiet
!pip install huggingface_hub --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00


# 2.Load Dataset



In [2]:
from datasets import load_dataset

dataset = load_dataset("rootsautomation/RICO-Screen2Words")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['screenId', 'captions', 'file_name', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'file_name_semantic', 'semantic_annotations', 'view_hierarchy', 'image', 'image_icon', 'image_semantic'],
        num_rows: 15743
    })
    val: Dataset({
        features: ['screenId', 'captions', 'file_name', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'file_name_semantic', 'semantic_annotations', 'view_hierarchy', 'image', 'image_icon', 'image_semantic'],
        num_rows: 2364
    })
    test: Dataset({
        features: ['screenId', 'captions', 'file_name', 'app_package_name', 'play_store_name', 'category', 'average_rating', 'number_of_ratings', 'number_of_downloads', 'file_name_icon', 'file_name_semantic', 'semantic_annotations', 'view_hierarchy', 'image', 'image_icon', 'imag

# Dataset Splits

The RICO Screen2Words dataset is divided into three splits:

- **Train** – used for model training  
- **Validation** – used to evaluate the model during training  
- **Test** – used for final qualitative evaluation

Using separate splits ensures that the model is evaluated on **data it has not seen during training**, which helps measure how well it generalizes to new UI screens.

---

# Fields Used
This project primarily uses:

| Field | Purpose |
|------|--------|
| **image** | Input UI screenshot |
| **captions** | Human-written description used as the training target |

Each screen may contain multiple captions. To create a single training pair, we use the **first caption**: This creates a consistent **image → caption** pair for training the model.


# 3. Loading the Pretrained BLIP Model

To perform image caption generation, we use the pretrained **BLIP (Bootstrapping Language Image Pretraining)** model from Hugging Face:

`Salesforce/blip-image-captioning-base`

### Why BLIP?

BLIP was selected for the following reasons:

- **Designed for vision–language tasks**  
  BLIP is specifically built for multimodal tasks such as image captioning and visual question answering, making it well suited for generating descriptions of UI screenshots.

- **Strong pretrained multimodal representations**  
  The model has already been trained on large image–text datasets, allowing it to understand relationships between visual features and language.

- **Efficient enough for T4 GPUs**  
  The base BLIP model is lightweight compared to many large vision–language models, making it practical to fine-tune on the T4 compute environment required for this assignment.

### Processor

We also load the **BLIP processor**, which prepares inputs for the model by:

- resizing and normalizing images
- tokenizing the caption text

Using a pretrained model and processor allows us to focus on **fine-tuning the model for the RICO Screen2Words dataset**, rather than training a multimodal model from scratch.


In [3]:
#load blip model
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [4]:
#check
sample = dataset["train"][0]

print(sample["captions"])
print(type(sample["image"]))#

['notification displaying a calendar with done option', 'pop up displaying a calendar', 'pop up displaying calendar to select date', 'pop up window of calendar with options', 'pop-up of a calendar to select date of birth']
<class 'PIL.JpegImagePlugin.JpegImageFile'>


In [13]:
train_dataset = dataset["train"]
val_dataset = dataset["val"]

# 4.Data Preprocessing

Before training the model, the dataset must be converted into the format expected by the BLIP model.

Each example in the dataset contains:

- an **image** (UI screenshot)
- a list of **captions** describing the screen

During preprocessing we perform the following steps:

1. Extract the screenshot from the `image` field.
2. Select a caption from the `captions` list.
3. Use the **BLIP processor** to convert the image and caption into model inputs.

### Tokenization and Image Processing

The BLIP processor performs:

- **image preprocessing** (resizing and normalization)
- **text tokenization** for the caption

### Labels for Training

For caption generation, the model must learn to predict the caption tokens.  
Therefore, we set:

labels = input_ids


This allows the model to compute the **language generation loss** during training.

### Padding and Truncation

We enable:

- `padding="max_length"`
- `truncation=True`

This ensures that all sequences have the same length, which allows them to be batched efficiently during training.


In [14]:
def preprocess(example):

    image = example["image"]
    caption = example["captions"][0]

    inputs = processor(
        images=image,
        text=caption,
        padding="max_length",
        truncation=True
    )

    inputs["labels"] = inputs["input_ids"]

    return inputs

### Applying Preprocessing to the Dataset

The preprocessing function is applied to both the training and validation splits using the Hugging Face `map()` function.

This converts each dataset example into the tensor format required by the model while removing unused columns from the original dataset.


In [15]:
train_dataset = train_dataset.map(
    preprocess,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    preprocess,
    remove_columns=val_dataset.column_names
)

In [16]:
train_dataset.set_format("torch")
val_dataset.set_format("torch")

#5. Data Collator

During training, the model processes examples in **batches** rather than one sample at a time.

After preprocessing, each dataset example contains tensors such as:

- `pixel_values`
- `input_ids`
- `attention_mask`
- `labels`

The **data collator** combines multiple examples into a single batch.  
This is done by stacking tensors using `torch.stack()` so that they can be processed efficiently by the model during training.

The collator returns a dictionary containing all required inputs for the BLIP model.

In [22]:
import torch

def collate_fn(batch):

    pixel_values = torch.stack([item["pixel_values"].squeeze(0) for item in batch])
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])

    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

Training Configuration

We fine-tune BLIP on the RICO-Screen2Words dataset.

Training settings:
 **Batch size: 4**  
  A small batch size is used to ensure the model fits within **T4 GPU memory constraints**.

- **Epochs: 3**  
  Multiple passes over the dataset allow the model to adapt to the RICO Screen2Words task while keeping training time reasonable.

- **Learning rate: 5e-5**  
  A commonly used learning rate for fine-tuning transformer-based models, allowing stable updates without disrupting pretrained knowledge.

- **FP16 mixed precision**  
  Enabled to reduce GPU memory usage and improve training speed on T4 GPUs.

In [24]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./rico-caption-model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_steps=100,
    save_steps=500,
    fp16=True,
    remove_unused_columns=False
)

In [25]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

#6. Model Training


The Hugging Face **Trainer** is used to manage the training process.

The Trainer handles:

- batching of data
- forward and backward passes
- optimization and loss computation
- periodic evaluation on the validation set

The model is trained to generate captions for UI screenshots by minimizing the difference between the **generated caption tokens and the ground-truth captions**.

After training, the model is evaluated on the validation dataset and the fine-tuned model weights are saved for later use.

In [26]:
trainer.train()

# evaluate
eval_results = trainer.evaluate()
print(eval_results)

# save model
trainer.save_model("rico-ui-caption-model")
processor.save_pretrained("rico-ui-caption-model")

{'eval_loss': 0.038679998368024826, 'eval_runtime': 151.7014, 'eval_samples_per_second': 15.583, 'eval_steps_per_second': 3.896, 'epoch': 3.0}


['rico-ui-caption-model/processor_config.json']

In [27]:
import shutil
from google.colab import files

shutil.make_archive("rico-ui-caption-model", "zip", "rico-ui-caption-model")
files.download("rico-ui-caption-model.zip")

In [28]:
import shutil
from google.colab import files

shutil.make_archive("rico-caption-model", "zip", "rico-caption-model")
files.download("rico-caption-model.zip")

##Push to Hugging Face

In [29]:
from huggingface_hub import notebook_login
notebook_login()

In [35]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("rico-ui-caption-model")
model = BlipForConditionalGeneration.from_pretrained("rico-ui-caption-model")

repo_id = "nikilovesml/temporal_trio-rico-screen2words-blip-caption"

model.push_to_hub(repo_id)
processor.push_to_hub(repo_id)

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


CommitInfo(commit_url='https://huggingface.co/nikilovesml/temporal_trio-rico-screen2words-blip-caption/commit/93fc99c8481a1053c33308e25489f08aa2c8a1a2', commit_message='Upload processor', commit_description='', oid='93fc99c8481a1053c33308e25489f08aa2c8a1a2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/nikilovesml/temporal_trio-rico-screen2words-blip-caption', endpoint='https://huggingface.co', repo_type='model', repo_id='nikilovesml/temporal_trio-rico-screen2words-blip-caption'), pr_revision=None, pr_num=None)

In [36]:
#check
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("nikilovesml/temporal_trio-rico-screen2words-blip-caption")
model = BlipForConditionalGeneration.from_pretrained("nikilovesml/temporal_trio-rico-screen2words-blip-caption")

print("Model loaded successfully!")

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded successfully!


#7. Evaluation

In [34]:
#Qualitative Evaluation (Ground Truth vs Prediction)
print("Sample Predictions:\n")

for i in range(5):

    sample = dataset["test"][i]
    image = sample["image"]

    inputs = processor(image, return_tensors="pt")

    output = model.generate(**inputs)

    prediction = processor.decode(output[0], skip_special_tokens=True)

    print("Ground Truth:", sample["captions"][0])
    print("Prediction:", prediction)
    print()

Sample Predictions:

Ground Truth: app showing the weight tracking page
Prediction: display of a page showing of online marketing app

Ground Truth: list of application options for browsing
Prediction: display of a pop up to open

Ground Truth: page asking to choose an app to share
Prediction: display of list of sharing applications

Ground Truth: display with multiple options
Prediction: display of a page showing of car maintenance app

Ground Truth: pop up displaying information
Prediction: pop up displaying about the app information



#8. Inference using HuggingFace Model

In [37]:
#Inference Using Model Downloaded from Hugging Face
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("nikilovesml/temporal_trio-rico-screen2words-blip-caption")
model = BlipForConditionalGeneration.from_pretrained("nikilovesml/temporal_trio-rico-screen2words-blip-caption")

for i in range(3):
    image = dataset["test"][i]["image"]

    inputs = processor(image, return_tensors="pt")

    out = model.generate(**inputs)

    caption = processor.decode(out[0], skip_special_tokens=True)

    print(f"Example {i+1}:")
    print("Generated Caption:", caption)
    print()

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Example 1:
Generated Caption: display of a page showing of online marketing app

Example 2:
Generated Caption: display of a pop up to open

Example 3:
Generated Caption: display of list of sharing applications



# Results and Discussion

### Training Performance

The BLIP model was fine-tuned for **3 epochs** on the RICO Screen2Words dataset using the Hugging Face Trainer API.

During training, the loss gradually decreased from approximately **0.04 to around 0.027**, indicating that the model successfully learned the relationship between UI screenshots and their textual descriptions.

Final evaluation metrics:

| Metric | Value |
|------|------|
| Validation Loss | 0.0387 |
| Training Epochs | 3 |
| Training Steps | 11,808 |

The relatively low validation loss suggests that the model generalizes reasonably well to unseen UI screens.

### Observations
- The training loss gradually decreased during training, indicating that the model was successfully learning to generate captions for UI screenshots.
- The final validation loss of **0.0387** suggests that the model was able to generalize reasonably well to unseen UI screens.
- No significant instability or divergence was observed during training.


---

### Qualitative Evaluation

We evaluated the model by generating captions for several unseen UI screenshots from the test set and comparing them with the ground-truth captions.

Example comparisons:

| Ground Truth | Model Prediction |
|------|------|
| app showing the weight tracking page | display of a page showing of online marketing app |
| list of application options for browsing | display of a pop up to open |
| page asking to choose an app to share | display of list of sharing applications |

Observations:

- The model generally captures the **overall structure and intent of the screen**.
- Predictions are often **semantically similar to the ground truth**, though wording may differ.
- Some captions are **more generic**, which is common for image captioning models.

Overall, the results demonstrate that the fine-tuned BLIP model can generate **reasonable summaries of mobile UI screenshots**.


# Observations

During training and evaluation, several observations were made.

### Loss Reduction

The training loss decreased steadily during training, suggesting that the model successfully learned the mapping between UI screenshots and captions.

### Caption Quality

Generated captions generally capture:

- the structure of the interface
- the presence of menus or lists
- the purpose of the screen

However, captions are sometimes **generic**, which is common for captioning models trained on relatively small datasets.

### Semantic Similarity

In many cases, predictions were **semantically similar to the ground truth** even if the wording differed.

Example:

Ground Truth:
page asking to choose an app to share

Prediction:
display of list of sharing applications

### Limitations

The model occasionally:

- misidentifies the application context
- generates vague descriptions

These limitations are expected given the dataset size and training duration.


# Conclusion

As a part of this Orange Problem, we successfully fine-tuned a multimodal BLIP model to generate captions for mobile UI screenshots using the RICO Screen2Words dataset.

The model learned to interpret UI layouts and produce short textual summaries of screen content.

Key outcomes of the project:

- Implemented a complete multimodal fine-tuning pipeline
- Trained a vision-language model on UI screenshot data
- Uploaded the trained model to Hugging Face
- Demonstrated reproducible inference using the Hugging Face Transformers library

The results show that pretrained multimodal models such as BLIP can be effectively adapted for specialized tasks like **UI screen summarization**.
